# Amazon Reviews 2023: análisis exploratorio

Este notebook muestra una forma segura y escalable de comenzar a analizar Amazon Reviews 2023 desde Google Colab.

## Qué vamos a hacer

1. Instalar las librerías.
2. Comparar las categorías del dataset.
3. Elegir una categoría.
4. Cargar muestras mediante **streaming**.
5. Explorar reviews y metadatos.
6. Crear gráficos.
7. Relacionar ambas tablas mediante `parent_asin`.
8. Exportar resultados pequeños a CSV.

> No descargaremos el dataset completo. El objetivo es construir primero un prototipo reproducible.


In [ ]:
# Instalamos las dependencias dentro del entorno temporal de Google Colab.
%pip install -q -U datasets huggingface_hub pandas matplotlib lxml


In [ ]:
from collections import Counter
from itertools import islice
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

DATASET_ID = "McAuley-Lab/Amazon-Reviews-2023"
DATASET_SITE = "https://amazon-reviews-2023.github.io/"

# Parámetros principales: cambia solamente estos valores al comenzar.
CATEGORY = "All_Beauty"
REVIEW_SAMPLE_SIZE = 5_000
METADATA_SAMPLE_SIZE = 3_000
SEED = 42

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Categoría seleccionada: {CATEGORY}")
print(f"Muestra de reviews: {REVIEW_SAMPLE_SIZE:,}")
print(f"Muestra de metadatos: {METADATA_SAMPLE_SIZE:,}")


## 1. Resumen de categorías

Primero intentamos leer la tabla oficial publicada por McAuley Lab. Si la estructura de la página cambia, utilizamos una copia de respaldo incluida en el notebook.


In [ ]:
CATEGORY_STATS_FALLBACK = [['All_Beauty', '632.0K', '112.6K', '701.5K'], ['Amazon_Fashion', '2.0M', '825.9K', '2.5M'], ['Appliances', '1.8M', '94.3K', '2.1M'], ['Arts_Crafts_and_Sewing', '4.6M', '801.3K', '9.0M'], ['Automotive', '8.0M', '2.0M', '20.0M'], ['Baby_Products', '3.4M', '217.7K', '6.0M'], ['Beauty_and_Personal_Care', '11.3M', '1.0M', '23.9M'], ['Books', '10.3M', '4.4M', '29.5M'], ['CDs_and_Vinyl', '1.8M', '701.7K', '4.8M'], ['Cell_Phones_and_Accessories', '11.6M', '1.3M', '20.8M'], ['Clothing_Shoes_and_Jewelry', '22.6M', '7.2M', '66.0M'], ['Digital_Music', '101.0K', '70.5K', '130.4K'], ['Electronics', '18.3M', '1.6M', '43.9M'], ['Gift_Cards', '132.7K', '1.1K', '152.4K'], ['Grocery_and_Gourmet_Food', '7.0M', '603.2K', '14.3M'], ['Handmade_Products', '586.6K', '164.7K', '664.2K'], ['Health_and_Household', '12.5M', '797.4K', '25.6M'], ['Health_and_Personal_Care', '461.7K', '60.3K', '494.1K'], ['Home_and_Kitchen', '23.2M', '3.7M', '67.4M'], ['Industrial_and_Scientific', '3.4M', '427.5K', '5.2M'], ['Kindle_Store', '5.6M', '1.6M', '25.6M'], ['Magazine_Subscriptions', '60.1K', '3.4K', '71.5K'], ['Movies_and_TV', '6.5M', '747.8K', '17.3M'], ['Musical_Instruments', '1.8M', '213.6K', '3.0M'], ['Office_Products', '7.6M', '710.4K', '12.8M'], ['Patio_Lawn_and_Garden', '8.6M', '851.7K', '16.5M'], ['Pet_Supplies', '7.8M', '492.7K', '16.8M'], ['Software', '2.6M', '89.2K', '4.9M'], ['Sports_and_Outdoors', '10.3M', '1.6M', '19.6M'], ['Subscription_Boxes', '15.2K', '641', '16.2K'], ['Tools_and_Home_Improvement', '12.2M', '1.5M', '27.0M'], ['Toys_and_Games', '8.1M', '890.7K', '16.3M'], ['Video_Games', '2.8M', '137.2K', '4.6M'], ['Unknown', '23.1M', '13.2M', '63.8M']]

def compact_to_number(value):
    """Convierte textos como 632.0K, 2.5M o 1.7B en números."""
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", "")
    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}

    if text and text[-1].upper() in multipliers:
        return float(text[:-1]) * multipliers[text[-1].upper()]

    return float(text)

def load_category_statistics():
    """Carga las estadísticas oficiales o usa el respaldo del notebook."""
    try:
        tables = pd.read_html(DATASET_SITE)
        table = next(
            table for table in tables
            if "Category" in table.columns and "#Rating" in table.columns
        )
        stats = table.rename(columns={
            "Category": "category",
            "#User": "users",
            "#Item": "items",
            "#Rating": "ratings",
        })[["category", "users", "items", "ratings"]]
        source = "tabla oficial en línea"
    except Exception as exc:
        stats = pd.DataFrame(
            CATEGORY_STATS_FALLBACK,
            columns=["category", "users", "items", "ratings"],
        )
        source = f"respaldo local del notebook: {type(exc).__name__}"

    for column in ["users", "items", "ratings"]:
        stats[column] = stats[column].apply(compact_to_number)

    return stats, source

category_stats, category_source = load_category_statistics()
print("Fuente utilizada:", category_source)
display(category_stats.head(10))


In [ ]:
# Gráfico 1: categorías con mayor cantidad de ratings.
top_ratings = category_stats.nlargest(15, "ratings").sort_values("ratings")

plt.figure(figsize=(11, 7))
plt.barh(top_ratings["category"], top_ratings["ratings"] / 1_000_000)
plt.title("Top 15 categorías por cantidad de ratings")
plt.xlabel("Ratings (millones)")
plt.ylabel("Categoría")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 2: categorías con mayor cantidad de ítems.
top_items = category_stats.nlargest(15, "items").sort_values("items")

plt.figure(figsize=(11, 7))
plt.barh(top_items["category"], top_items["items"] / 1_000_000)
plt.title("Top 15 categorías por cantidad de ítems")
plt.xlabel("Ítems (millones)")
plt.ylabel("Categoría")
plt.tight_layout()
plt.show()


## 2. Función para cargar una muestra por streaming

`streaming=True` permite comenzar a iterar los registros sin descargar primero todos los archivos.

La muestra se mezcla de forma aproximada con un búfer para evitar utilizar únicamente los primeros registros del archivo.


In [ ]:
def stream_sample(config_name, sample_size, seed=42, buffer_size=10_000):
    """Carga una muestra de una configuración de Amazon Reviews 2023."""
    print(f"Cargando configuración: {config_name}")

    # En las versiones actuales de datasets, las configuraciones Parquet
    # normalmente funcionan sin trust_remote_code.
    try:
        stream = load_dataset(
            DATASET_ID,
            config_name,
            split="full",
            streaming=True,
        )
    except Exception as first_error:
        print("Primer intento falló. Probando el cargador histórico del dataset...")
        try:
            stream = load_dataset(
                DATASET_ID,
                config_name,
                split="full",
                streaming=True,
                trust_remote_code=True,
            )
        except Exception:
            raise RuntimeError(
                f"No se pudo cargar {config_name}. Error inicial: {first_error}"
            ) from first_error

    # Shuffle en streaming es aproximado; no carga todo el dataset en RAM.
    stream = stream.shuffle(
        seed=seed,
        buffer_size=max(buffer_size, sample_size),
    )

    rows = list(islice(stream, sample_size))
    if not rows:
        raise ValueError(f"La configuración {config_name} no devolvió registros.")

    return pd.DataFrame(rows)


## 3. Cargar y revisar las reviews

Campos importantes:

- `rating`: estrellas otorgadas.
- `title` y `text`: contenido de la reseña.
- `parent_asin`: identificador para conectar con los metadatos.
- `timestamp`: fecha de la interacción.
- `verified_purchase`: indica compra verificada.
- `helpful_vote`: votos de utilidad.


In [ ]:
reviews_config = f"raw_review_{CATEGORY}"
reviews_df = stream_sample(
    config_name=reviews_config,
    sample_size=REVIEW_SAMPLE_SIZE,
    seed=SEED,
)

# Transformaciones básicas.
reviews_df["rating"] = pd.to_numeric(reviews_df["rating"], errors="coerce")
reviews_df["helpful_vote"] = pd.to_numeric(
    reviews_df["helpful_vote"], errors="coerce"
).fillna(0)

# El timestamp del dataset está expresado en milisegundos Unix.
reviews_df["review_date"] = pd.to_datetime(
    reviews_df["timestamp"],
    unit="ms",
    errors="coerce",
)

review_columns = [
    "parent_asin",
    "rating",
    "title",
    "verified_purchase",
    "helpful_vote",
    "review_date",
]
display(reviews_df[review_columns].head(10))
print("Dimensión de la muestra:", reviews_df.shape)


In [ ]:
# Calidad y valores faltantes de la muestra.
review_quality = pd.DataFrame({
    "tipo": reviews_df.dtypes.astype(str),
    "nulos": reviews_df.isna().sum(),
    "porcentaje_nulos": (reviews_df.isna().mean() * 100).round(2),
}).sort_values("porcentaje_nulos", ascending=False)

display(review_quality)


In [ ]:
# Gráfico 3: distribución de estrellas.
rating_counts = reviews_df["rating"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(rating_counts.index.astype(str), rating_counts.values)
plt.title(f"Distribución de ratings — {CATEGORY}")
plt.xlabel("Estrellas")
plt.ylabel("Cantidad de reviews en la muestra")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 4: compras verificadas.
verified_counts = (
    reviews_df["verified_purchase"]
    .fillna(False)
    .value_counts()
    .rename(index={True: "Verificada", False: "No verificada"})
)

plt.figure(figsize=(7, 5))
plt.bar(verified_counts.index.astype(str), verified_counts.values)
plt.title(f"Compras verificadas — {CATEGORY}")
plt.xlabel("Tipo de compra")
plt.ylabel("Cantidad de reviews")
plt.tight_layout()
plt.show()


In [ ]:
# Tabla y gráfico 5: productos más reseñados dentro de la muestra.
top_reviewed_items = (
    reviews_df.groupby("parent_asin")
    .agg(
        review_count=("parent_asin", "size"),
        average_rating=("rating", "mean"),
        helpful_votes=("helpful_vote", "sum"),
    )
    .sort_values("review_count", ascending=False)
    .head(15)
    .reset_index()
)

display(top_reviewed_items)

plot_items = top_reviewed_items.sort_values("review_count")
plt.figure(figsize=(10, 7))
plt.barh(plot_items["parent_asin"], plot_items["review_count"])
plt.title(f"Ítems más reseñados en la muestra — {CATEGORY}")
plt.xlabel("Cantidad de reviews")
plt.ylabel("parent_asin")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico 6: evolución temporal de las reviews por año.
reviews_by_year = (
    reviews_df.dropna(subset=["review_date"])
    .assign(year=lambda df: df["review_date"].dt.year)
    .groupby("year")
    .size()
)

plt.figure(figsize=(11, 5))
plt.plot(reviews_by_year.index, reviews_by_year.values, marker="o")
plt.title(f"Reviews por año en la muestra — {CATEGORY}")
plt.xlabel("Año")
plt.ylabel("Cantidad de reviews")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 4. Cargar los metadatos de productos

Los metadatos contienen nombre, tienda, precio, calificación promedio, cantidad total de ratings, características y descripción.


In [ ]:
metadata_config = f"raw_meta_{CATEGORY}"
metadata_df = stream_sample(
    config_name=metadata_config,
    sample_size=METADATA_SAMPLE_SIZE,
    seed=SEED,
)

metadata_df["average_rating"] = pd.to_numeric(
    metadata_df["average_rating"], errors="coerce"
)
metadata_df["rating_number"] = pd.to_numeric(
    metadata_df["rating_number"], errors="coerce"
)

# Algunos registros históricos pueden representar la ausencia de precio
# mediante None, "None", símbolos o textos.
metadata_df["price_numeric"] = pd.to_numeric(
    metadata_df["price"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .replace("", np.nan),
    errors="coerce",
)

metadata_columns = [
    "parent_asin",
    "title",
    "store",
    "average_rating",
    "rating_number",
    "price_numeric",
    "main_category",
]
display(metadata_df[metadata_columns].head(10))
print("Dimensión de la muestra:", metadata_df.shape)


In [ ]:
# Tiendas más frecuentes dentro de la muestra de metadatos.
top_stores = (
    metadata_df["store"]
    .dropna()
    .astype(str)
    .replace("", np.nan)
    .dropna()
    .value_counts()
    .head(15)
    .sort_values()
)

plt.figure(figsize=(10, 7))
plt.barh(top_stores.index, top_stores.values)
plt.title(f"Tiendas más frecuentes en la muestra — {CATEGORY}")
plt.xlabel("Cantidad de productos")
plt.ylabel("Tienda")
plt.tight_layout()
plt.show()


In [ ]:
# Precio frente a calificación promedio.
price_rating = metadata_df.dropna(
    subset=["price_numeric", "average_rating"]
).copy()

# Eliminamos valores extremos para que el gráfico sea legible.
if not price_rating.empty:
    upper_price = price_rating["price_numeric"].quantile(0.99)
    price_rating = price_rating[
        price_rating["price_numeric"].between(0, upper_price)
    ]

plt.figure(figsize=(9, 6))
plt.scatter(
    price_rating["price_numeric"],
    price_rating["average_rating"],
    alpha=0.35,
)
plt.title(f"Precio frente a calificación promedio — {CATEGORY}")
plt.xlabel("Precio en USD")
plt.ylabel("Calificación promedio")
plt.tight_layout()
plt.show()

display(
    price_rating[
        ["parent_asin", "title", "price_numeric", "average_rating", "rating_number"]
    ].head(10)
)


## 5. Conectar reviews con metadatos

La unión correcta es:

```text
reviews.parent_asin = metadata.parent_asin
```

Como ambas tablas son muestras independientes, no todos los productos necesariamente aparecerán en las dos. Primero hacemos una unión directa con las coincidencias disponibles.


In [ ]:
item_review_summary = (
    reviews_df.groupby("parent_asin")
    .agg(
        sample_review_count=("parent_asin", "size"),
        sample_average_rating=("rating", "mean"),
        sample_helpful_votes=("helpful_vote", "sum"),
    )
    .reset_index()
)

items_joined = item_review_summary.merge(
    metadata_df[
        [
            "parent_asin",
            "title",
            "store",
            "average_rating",
            "rating_number",
            "price_numeric",
        ]
    ].drop_duplicates("parent_asin"),
    on="parent_asin",
    how="left",
)

items_joined = items_joined.sort_values(
    "sample_review_count", ascending=False
)

display(items_joined.head(20))
print(
    "Productos con metadatos encontrados:",
    items_joined["title"].notna().sum(),
    "de",
    len(items_joined),
)


### Búsqueda dirigida opcional

Si faltan los metadatos de los productos más reseñados, podemos recorrer el stream de metadatos hasta encontrarlos. La búsqueda se limita para evitar procesos demasiado largos en categorías enormes.


In [ ]:
def find_metadata_for_asins(category, target_asins, max_scan=100_000):
    """Busca metadatos de ASIN concretos sin descargar toda la categoría."""
    target_asins = set(target_asins)
    found = []
    scanned = 0

    stream = load_dataset(
        DATASET_ID,
        f"raw_meta_{category}",
        split="full",
        streaming=True,
    )

    for row in stream:
        scanned += 1
        if row.get("parent_asin") in target_asins:
            found.append(row)
            target_asins.remove(row["parent_asin"])

        if not target_asins or scanned >= max_scan:
            break

    print(f"Registros inspeccionados: {scanned:,}")
    print(f"Productos encontrados: {len(found):,}")
    print(f"Productos pendientes: {len(target_asins):,}")
    return pd.DataFrame(found)

# Descomenta estas líneas para buscar los metadatos de los 10 productos
# con más reviews dentro de la muestra.
#
# target_asins = top_reviewed_items["parent_asin"].head(10)
# directed_metadata_df = find_metadata_for_asins(
#     CATEGORY,
#     target_asins,
#     max_scan=100_000,
# )
# display(directed_metadata_df[["parent_asin", "title", "store", "price"]])


## 6. Exportar resultados

Exportamos solamente tablas procesadas y pequeñas. No exportamos el dataset bruto.


In [ ]:
category_stats.to_csv(
    OUTPUT_DIR / "category_statistics_clean.csv",
    index=False,
)
top_reviewed_items.to_csv(
    OUTPUT_DIR / f"top_reviewed_items_{CATEGORY}.csv",
    index=False,
)
items_joined.head(1_000).to_csv(
    OUTPUT_DIR / f"items_reviews_join_{CATEGORY}.csv",
    index=False,
)

print("Archivos generados:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path)


## Próximos análisis recomendados

Cuando este notebook funcione correctamente, el proyecto puede crecer en tres etapas:

1. **Análisis descriptivo**
   - categorías;
   - precios;
   - ratings;
   - tiendas;
   - evolución temporal.

2. **Construcción de indicadores de éxito**
   - cantidad de ratings;
   - rating promedio;
   - proporción de compras verificadas;
   - votos útiles;
   - crecimiento temporal;
   - popularidad relativa dentro de la categoría.

3. **Modelo predictivo**
   - preparar variables de precio, categoría y texto;
   - definir una variable objetivo;
   - separar entrenamiento y prueba;
   - entrenar un modelo base;
   - evaluar si existe una señal predictiva real.

### Advertencia metodológica

Una muestra de streaming sirve para explorar y validar el código, pero no debe presentarse como una estimación exacta de toda la categoría sin estudiar el método de muestreo y sus posibles sesgos.
